# EXP 01 — Baseline Match-Level Common-Features-Only

Eksperimen ini dibuat sebagai baseline yang **aman untuk inference**, **rapi**, dan **langsung bisa dipakai** untuk menghasilkan prediksi pada `test.csv`.

## Tujuan
Target yang diprediksi:
- `team_goals`
- `opp_goals`

## Prinsip desain EXP 01
1. Data asli berbentuk **2 row per match**, jadi kita ubah dulu menjadi **1 row per match**.
2. Model final **hanya memakai fitur yang tersedia di train dan test**.
3. Validasi harus **strict time-based**, bukan random split.
4. Notebook ini sengaja dijaga tetap sehat dulu: cukup informatif, cukup ketat, dan tidak terlalu rumit.

In [ ]:

# 01. Setup & import

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

SEED = 42
MAX_GOALS = 31
DATA_DIR = Path(".")

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("Seed:", SEED)
print("Working directory:", DATA_DIR.resolve())

In [ ]:

# 02. Load data

train_path = DATA_DIR / "train.csv"
test_path  = DATA_DIR / "test.csv"

assert train_path.exists(), f"train.csv tidak ditemukan di {train_path.resolve()}"
assert test_path.exists(), f"test.csv tidak ditemukan di {test_path.resolve()}"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("train shape:", train.shape)
print("test shape :", test.shape)
display(train.head(3))
display(test.head(3))

# 03. Ringkasan insight dari EDA yang relevan untuk EXP 01

Dari EDA sebelumnya, ada beberapa poin yang paling penting untuk baseline ini:

- Format data memang **2 row per match** dan struktur pasangannya konsisten.
- Train dan test dipisahkan berdasarkan waktu, jadi validasi **wajib time-based**.
- Ada **train–test shift** yang nyata, termasuk pergeseran distribusi `gender`.
- Banyak fitur yang hanya ada di train, jadi untuk baseline awal ini kita **sengaja membatasi diri ke common features**.
- Ada missing value yang cukup besar di beberapa kolom common.
- Nilai sentinel `-9999` pada `altitude_venue` harus diperlakukan sebagai **missing**, bukan angka asli.
- Karena output akhir tetap row-level (`Id`, `team_goals`, `opp_goals`), maka setelah modeling di level match, prediksi harus dikembalikan lagi ke format row-level secara konsisten.

In [ ]:

# 04. Audit struktur train/test dan common features

TARGET_COLS = ["team_goals", "opp_goals"]
ID_COLS = ["Id", "match_id"]

common_cols = sorted(set(train.columns) & set(test.columns))
train_only_cols = sorted(set(train.columns) - set(test.columns))
test_only_cols = sorted(set(test.columns) - set(train.columns))

print("Jumlah kolom common    :", len(common_cols))
print("Jumlah kolom train-only:", len(train_only_cols))
print("Jumlah kolom test-only :", len(test_only_cols))

print("\nCommon columns:")
print(common_cols)

print("\nTrain-only columns:")
print(train_only_cols)

assert len(test_only_cols) == 0, "Ekspektasi EXP 01: test tidak punya kolom unik tambahan."

train_pair_counts = train["match_id"].value_counts()
test_pair_counts = test["match_id"].value_counts()

assert train_pair_counts.eq(2).all(), "Train tidak konsisten 2 row per match."
assert test_pair_counts.eq(2).all(), "Test tidak konsisten 2 row per match."

print("\nSemua match di train dan test punya tepat 2 baris.")
print("Jumlah match train:", train["match_id"].nunique())
print("Jumlah match test :", test["match_id"].nunique())

train["date"] = pd.to_datetime(train["date"], errors="coerce")
test["date"] = pd.to_datetime(test["date"], errors="coerce")

assert train["date"].notna().all(), "Ada date train yang gagal diparse."
assert test["date"].notna().all(), "Ada date test yang gagal diparse."

print("\nRentang waktu train:", train["date"].min().date(), "s/d", train["date"].max().date())
print("Rentang waktu test :", test["date"].min().date(), "s/d", test["date"].max().date())
print("Apakah test mulai setelah train berakhir?", bool(test["date"].min() > train["date"].max()))

print("\nProporsi gender train:")
display((train["gender"].value_counts(normalize=True) * 100).round(2).rename("pct"))

print("Proporsi gender test:")
display((test["gender"].value_counts(normalize=True) * 100).round(2).rename("pct"))

# 05. Transformasi ke match-level

Aturan deterministik yang dipakai di notebook ini:

1. Untuk setiap `match_id`, dua baris diurutkan dengan:
   - `is_home` **descending** (baris home lebih dulu),
   - lalu `Id` ascending sebagai tie-break aman.
2. Setelah itu:
   - baris pertama dianggap **side home**
   - baris kedua dianggap **side away**
3. Target match-level diambil dari baris home:
   - `home_goals = team_goals`
   - `away_goals = opp_goals`

Kenapa aturan ini aman?
- Struktur file memang konsisten 2 row per match.
- `team` dan `opponent` saling mirror antar dua baris.
- Dengan aturan tetap ini, transformasi train dan test menjadi konsisten.

In [ ]:

# 05. Helper function untuk ubah row-level -> match-level

COMMON_BASE_COLS = [
    "Id",
    "match_id",
    "date",
    "gender",
    "team",
    "opponent",
    "is_home",
    "neutral",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
    "population_team",
    "population_opp",
    "gdp_per_capita_team",
    "gdp_per_capita_opp",
    "altitude_venue",
    "distance_travel_team",
    "distance_travel_opp",
    "temperature_venue",
]

def build_match_level(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    work = df.copy()

    # parsing date & cleaning sentinel
    work["date"] = pd.to_datetime(work["date"], errors="coerce")
    work.loc[work["altitude_venue"] == -9999, "altitude_venue"] = np.nan

    sort_cols = ["match_id", "is_home", "Id"]
    work = work.sort_values(sort_cols, ascending=[True, False, True]).reset_index(drop=True)

    pair_counts = work["match_id"].value_counts().sort_index()
    assert pair_counts.eq(2).all(), "Ada match_id yang tidak punya tepat 2 baris."

    home = work.iloc[::2].reset_index(drop=True).copy()
    away = work.iloc[1::2].reset_index(drop=True).copy()

    assert np.array_equal(home["match_id"].values, away["match_id"].values), "Pairing match_id tidak sinkron."
    assert home["is_home"].eq(1).all(), "Baris pertama pair seharusnya home."
    assert away["is_home"].eq(0).all(), "Baris kedua pair seharusnya away."
    assert (home["team"].values == away["opponent"].values).all(), "Mirror team-opponent tidak konsisten."
    assert (away["team"].values == home["opponent"].values).all(), "Mirror opponent-team tidak konsisten."

    # metadata yang harus identik di dalam 1 match
    stable_cols = ["date", "gender", "neutral", "tournament", "venue_country", "altitude_venue", "temperature_venue"]
    for col in stable_cols:
        left = home[col].astype(str).fillna("__nan__")
        right = away[col].astype(str).fillna("__nan__")
        assert (left.values == right.values).all(), f"Kolom {col} ternyata tidak stabil per match."

    match_df = pd.DataFrame({
        "match_id": home["match_id"],
        "date": home["date"],
        "home_id": home["Id"],
        "away_id": away["Id"],
        "gender": home["gender"],
        "neutral": home["neutral"].astype("int8"),
        "tournament": home["tournament"],
        "venue_country": home["venue_country"],
        "home_team": home["team"],
        "away_team": away["team"],
        "home_confederation": home["confederation_team"],
        "away_confederation": away["confederation_team"],
        "population_home": home["population_team"],
        "population_away": away["population_team"],
        "gdp_per_capita_home": home["gdp_per_capita_team"],
        "gdp_per_capita_away": away["gdp_per_capita_team"],
        "distance_travel_home": home["distance_travel_team"],
        "distance_travel_away": away["distance_travel_team"],
        "temperature_venue": home["temperature_venue"],
        "altitude_venue": home["altitude_venue"],
    })

    if is_train:
        assert {"team_goals", "opp_goals"}.issubset(work.columns), "Train harus punya target."
        assert np.array_equal(home["team_goals"].values, away["opp_goals"].values), "Mirror target tidak konsisten."
        assert np.array_equal(home["opp_goals"].values, away["team_goals"].values), "Mirror target tidak konsisten."

        match_df["home_goals"] = home["team_goals"].astype(float)
        match_df["away_goals"] = home["opp_goals"].astype(float)

    return match_df

train_match = build_match_level(train, is_train=True)
test_match  = build_match_level(test, is_train=False)

print("train_match shape:", train_match.shape)
print("test_match shape :", test_match.shape)
display(train_match.head(3))
display(test_match.head(3))

# 06. Cleaning dan preprocessing

Karena EXP 01 ini pakai model tree-based, strategi cleaning-nya sengaja sederhana dan aman:

- `date` diparse dengan aman.
- `altitude_venue == -9999` diubah ke `NaN`.
- Missing value **tidak dipaksa diimputasi agresif**; untuk tree model seperti CatBoost, missing bisa dibiarkan sebagai missing.
- Kita tambahkan beberapa **missing indicators** supaya model bisa membedakan nilai yang benar-benar kosong.

In [ ]:

# 06. Feature engineering yang tetap inference-safe
# Semua fitur turunan di bawah hanya berasal dari kolom common train-test.

def add_safe_features(match_df: pd.DataFrame) -> pd.DataFrame:
    df = match_df.copy()

    df["year"] = df["date"].dt.year.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype("int8")

    df["era_bucket"] = pd.cut(
        df["year"],
        bins=[0, 1949, 1979, 1999, 2009, 2100],
        labels=["pre_1950", "1950_1979", "1980_1999", "2000_2009", "2010_plus"]
    ).astype(str)

    for base in ["population", "gdp_per_capita", "distance_travel"]:
        df[f"{base}_diff"] = df[f"{base}_home"] - df[f"{base}_away"]
        df[f"{base}_abs_diff"] = (df[f"{base}_home"] - df[f"{base}_away"]).abs()
        df[f"{base}_mean"] = df[[f"{base}_home", f"{base}_away"]].mean(axis=1)

    df["same_confederation"] = (df["home_confederation"] == df["away_confederation"]).astype("int8")
    df["is_friendly"] = df["tournament"].str.lower().eq("friendly").astype("int8")

    missing_flag_cols = [
        "altitude_venue",
        "temperature_venue",
        "population_home",
        "population_away",
        "gdp_per_capita_home",
        "gdp_per_capita_away",
        "distance_travel_home",
        "distance_travel_away",
    ]

    for col in missing_flag_cols:
        df[f"{col}_missing"] = df[col].isna().astype("int8")

    return df

train_feat = add_safe_features(train_match)
test_feat  = add_safe_features(test_match)

print("Jumlah kolom train_feat:", train_feat.shape[1])
print("Jumlah kolom test_feat :", test_feat.shape[1])

# 07. Feature preparation

Untuk modeling, kita sengaja **tidak** memasukkan:
- `match_id`
- `date` mentah
- `home_id`, `away_id`
- target

Fitur kategorikal tetap dipertahankan sebagai kategorikal/string, karena CatBoost bisa menanganinya langsung.

In [ ]:

# 07. Menentukan feature set final untuk modeling

DROP_FOR_MODEL = {
    "match_id",
    "date",
    "home_id",
    "away_id",
    "home_goals",
    "away_goals",
}

feature_cols = [c for c in train_feat.columns if c not in DROP_FOR_MODEL]

categorical_cols = [
    "gender",
    "tournament",
    "venue_country",
    "home_team",
    "away_team",
    "home_confederation",
    "away_confederation",
    "era_bucket",
]

numeric_cols = [c for c in feature_cols if c not in categorical_cols]

for df in [train_feat, test_feat]:
    for col in categorical_cols:
        df[col] = df[col].astype(str).fillna("__nan__")

print("Jumlah feature final:", len(feature_cols))
print("Jumlah feature kategorikal:", len(categorical_cols))
print("Jumlah feature numerik:", len(numeric_cols))

display(pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(train_feat[c].dtype) for c in feature_cols],
    "is_categorical": [c in categorical_cols for c in feature_cols],
}).head(20))

# 08. Time-based validation design

Validasi dilakukan dengan prinsip berikut:

- Data match-level diurutkan berdasarkan `date`.
- Kita ambil **holdout 20% paling akhir** sebagai validation.
- Split dilakukan berdasarkan **tanggal cutoff**, jadi semua match pada tanggal validasi tetap masuk validation, dan tidak ada leakage tanggal yang tercampur.

Alasan desain ini:
- Test memang datang dari masa yang lebih akhir dibanding train.
- Random split akan terlalu optimistis.
- Untuk baseline awal, holdout waktu yang sederhana tapi ketat sudah cukup representatif.

In [ ]:

# 08. Membuat strict time-based holdout

train_feat = train_feat.sort_values(["date", "match_id"]).reset_index(drop=True)

cut_idx = int(len(train_feat) * 0.80)
cutoff_date = train_feat.loc[cut_idx, "date"].normalize()

train_mask = train_feat["date"] < cutoff_date
valid_mask = train_feat["date"] >= cutoff_date

train_part = train_feat.loc[train_mask].copy()
valid_part = train_feat.loc[valid_mask].copy()

assert train_part["date"].max() < valid_part["date"].min(), "Ada overlap waktu antara train dan valid."

print("Cutoff date :", cutoff_date.date())
print("Train part  :", train_part.shape)
print("Valid part  :", valid_part.shape)
print("Train range :", train_part['date'].min().date(), "s/d", train_part['date'].max().date())
print("Valid range :", valid_part['date'].min().date(), "s/d", valid_part['date'].max().date())

# 09. Baseline modeling

Di EXP 01 ini kita bandingkan dua baseline:

1. **Naive grouped baseline**  
   Prediksi dibuat dari rata-rata historis berdasarkan kombinasi kategori sederhana yang inference-safe.

2. **CatBoost Regressor**  
   Model tree-based yang nyaman untuk campuran numerik + kategorikal, dan tetap aman dipakai karena seluruh fitur berasal dari common features.

CatBoost dilatih terpisah untuk:
- `home_goals`
- `away_goals`

In [ ]:

# 09A. Helper evaluasi

def joint_mae(y_true_home, y_pred_home, y_true_away, y_pred_away):
    mae_home = mean_absolute_error(y_true_home, y_pred_home)
    mae_away = mean_absolute_error(y_true_away, y_pred_away)
    return mae_home, mae_away, (mae_home + mae_away) / 2

def exact_score_accuracy(y_true_home, y_pred_home, y_true_away, y_pred_away):
    return np.mean(
        (np.asarray(y_true_home) == np.asarray(y_pred_home)) &
        (np.asarray(y_true_away) == np.asarray(y_pred_away))
    )

def clip_and_round(x, low=0, high=MAX_GOALS):
    return np.clip(np.rint(x), low, high).astype(int)

In [ ]:

# 09B. Baseline 1 - naive grouped mean

GROUP_COLS = ["gender", "tournament", "neutral", "home_confederation", "away_confederation"]

global_home_mean = train_part["home_goals"].mean()
global_away_mean = train_part["away_goals"].mean()

group_stats = (
    train_part
    .groupby(GROUP_COLS, dropna=False)[["home_goals", "away_goals"]]
    .mean()
    .reset_index()
    .rename(columns={
        "home_goals": "pred_home",
        "away_goals": "pred_away",
    })
)

valid_naive = valid_part.merge(group_stats, on=GROUP_COLS, how="left")
valid_naive["pred_home"] = valid_naive["pred_home"].fillna(global_home_mean)
valid_naive["pred_away"] = valid_naive["pred_away"].fillna(global_away_mean)

naive_home_pred = clip_and_round(valid_naive["pred_home"])
naive_away_pred = clip_and_round(valid_naive["pred_away"])

naive_mae_home, naive_mae_away, naive_joint = joint_mae(
    valid_part["home_goals"], naive_home_pred,
    valid_part["away_goals"], naive_away_pred
)

naive_exact = exact_score_accuracy(
    valid_part["home_goals"], naive_home_pred,
    valid_part["away_goals"], naive_away_pred
)

print("Naive baseline selesai.")
print("MAE home :", round(naive_mae_home, 4))
print("MAE away :", round(naive_mae_away, 4))
print("Joint MAE:", round(naive_joint, 4))
print("Exact score acc:", round(naive_exact, 4))

In [ ]:

# 09C. Baseline 2 - CatBoost

X_train = train_part[feature_cols].copy()
X_valid = valid_part[feature_cols].copy()

y_train_home = train_part["home_goals"].copy()
y_train_away = train_part["away_goals"].copy()
y_valid_home = valid_part["home_goals"].copy()
y_valid_away = valid_part["away_goals"].copy()

cat_feature_idx = [feature_cols.index(col) for col in categorical_cols]

cat_params = dict(
    loss_function="Poisson",
    eval_metric="MAE",
    iterations=200,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    verbose=False,
)

cat_home = CatBoostRegressor(**cat_params)
cat_away = CatBoostRegressor(**cat_params)

cat_home.fit(
    X_train, y_train_home,
    cat_features=cat_feature_idx,
    eval_set=(X_valid, y_valid_home),
    use_best_model=True,
)

cat_away.fit(
    X_train, y_train_away,
    cat_features=cat_feature_idx,
    eval_set=(X_valid, y_valid_away),
    use_best_model=True,
)

cat_home_pred_float = np.clip(cat_home.predict(X_valid), 0, MAX_GOALS)
cat_away_pred_float = np.clip(cat_away.predict(X_valid), 0, MAX_GOALS)

cat_home_pred = clip_and_round(cat_home_pred_float)
cat_away_pred = clip_and_round(cat_away_pred_float)

cat_mae_home, cat_mae_away, cat_joint = joint_mae(
    y_valid_home, cat_home_pred,
    y_valid_away, cat_away_pred
)

cat_exact = exact_score_accuracy(
    y_valid_home, cat_home_pred,
    y_valid_away, cat_away_pred
)

print("CatBoost baseline selesai.")
print("Best iteration home:", cat_home.get_best_iteration())
print("Best iteration away:", cat_away.get_best_iteration())
print("MAE home :", round(cat_mae_home, 4))
print("MAE away :", round(cat_mae_away, 4))
print("Joint MAE:", round(cat_joint, 4))
print("Exact score acc:", round(cat_exact, 4))

# 10. Validation results

Di bagian ini kita tampilkan:
- tabel perbandingan baseline
- contoh prediksi vs aktual
- distribusi error sederhana
- performa pada match dengan total gol rendah vs lebih tinggi

In [ ]:

# 10A. Ringkasan perbandingan model

results_df = pd.DataFrame([
    {
        "model": "Naive grouped mean",
        "mae_home": naive_mae_home,
        "mae_away": naive_mae_away,
        "joint_mae": naive_joint,
        "exact_score_acc": naive_exact,
    },
    {
        "model": "CatBoost",
        "mae_home": cat_mae_home,
        "mae_away": cat_mae_away,
        "joint_mae": cat_joint,
        "exact_score_acc": cat_exact,
    },
]).sort_values("joint_mae").reset_index(drop=True)

display(results_df)

best_model_name = results_df.loc[0, "model"]
print("Best validation model:", best_model_name)

In [ ]:

# 10B. Contoh prediksi vs aktual pada validation set

valid_compare = valid_part[[
    "match_id", "date", "home_team", "away_team", "tournament", "gender",
    "home_goals", "away_goals"
]].copy()

valid_compare["pred_home_naive"] = naive_home_pred
valid_compare["pred_away_naive"] = naive_away_pred
valid_compare["pred_home_cat"] = cat_home_pred
valid_compare["pred_away_cat"] = cat_away_pred

valid_compare["abs_err_total_naive"] = (
    (valid_compare["home_goals"] - valid_compare["pred_home_naive"]).abs() +
    (valid_compare["away_goals"] - valid_compare["pred_away_naive"]).abs()
)

valid_compare["abs_err_total_cat"] = (
    (valid_compare["home_goals"] - valid_compare["pred_home_cat"]).abs() +
    (valid_compare["away_goals"] - valid_compare["pred_away_cat"]).abs()
)

display(valid_compare.head(15))

In [ ]:

# 10C. Distribusi error CatBoost

valid_compare["total_goals_true"] = valid_compare["home_goals"] + valid_compare["away_goals"]
valid_compare["total_goals_pred_cat"] = valid_compare["pred_home_cat"] + valid_compare["pred_away_cat"]
valid_compare["total_abs_error_cat"] = (
    (valid_compare["home_goals"] - valid_compare["pred_home_cat"]).abs() +
    (valid_compare["away_goals"] - valid_compare["pred_away_cat"]).abs()
)

plt.figure()
valid_compare["total_abs_error_cat"].hist(bins=20)
plt.title("Distribusi total absolute error - CatBoost")
plt.xlabel("Total absolute error")
plt.ylabel("Jumlah match")
plt.show()

segment_rows = []
for label, mask in {
    "low_total_goals_(<=2)": valid_compare["total_goals_true"] <= 2,
    "higher_total_goals_(>2)": valid_compare["total_goals_true"] > 2,
}.items():
    seg = valid_compare.loc[mask]
    segment_rows.append({
        "segment": label,
        "n_matches": len(seg),
        "mae_home_cat": mean_absolute_error(seg["home_goals"], seg["pred_home_cat"]) if len(seg) else np.nan,
        "mae_away_cat": mean_absolute_error(seg["away_goals"], seg["pred_away_cat"]) if len(seg) else np.nan,
        "joint_mae_cat": (
            mean_absolute_error(seg["home_goals"], seg["pred_home_cat"]) +
            mean_absolute_error(seg["away_goals"], seg["pred_away_cat"])
        ) / 2 if len(seg) else np.nan,
    })

segment_df = pd.DataFrame(segment_rows)
display(segment_df)

# 11. Final training

Karena target akhir notebook ini adalah membuat submission, model terbaik berdasarkan validation akan dilatih ulang pada **seluruh train match-level**.

Untuk menjaga notebook tetap sederhana dan stabil:
- jika model terbaik adalah CatBoost, kita fit ulang dua model CatBoost pada full training data
- jika model terbaik ternyata naive baseline, maka full-data grouped mean dipakai langsung

In [ ]:

# 11. Final training full data

X_full = train_feat[feature_cols].copy()
X_test = test_feat[feature_cols].copy()

y_full_home = train_feat["home_goals"].copy()
y_full_away = train_feat["away_goals"].copy()

final_artifacts = {}

if best_model_name == "CatBoost":
    final_home = CatBoostRegressor(**cat_params)
    final_away = CatBoostRegressor(**cat_params)

    final_home.fit(X_full, y_full_home, cat_features=cat_feature_idx, verbose=False)
    final_away.fit(X_full, y_full_away, cat_features=cat_feature_idx, verbose=False)

    test_home_pred = clip_and_round(np.clip(final_home.predict(X_test), 0, MAX_GOALS))
    test_away_pred = clip_and_round(np.clip(final_away.predict(X_test), 0, MAX_GOALS))

    final_artifacts["model"] = "CatBoost"
else:
    full_global_home = train_feat["home_goals"].mean()
    full_global_away = train_feat["away_goals"].mean()

    full_group_stats = (
        train_feat
        .groupby(GROUP_COLS, dropna=False)[["home_goals", "away_goals"]]
        .mean()
        .reset_index()
        .rename(columns={"home_goals": "pred_home", "away_goals": "pred_away"})
    )

    test_naive = test_feat.merge(full_group_stats, on=GROUP_COLS, how="left")
    test_home_pred = clip_and_round(test_naive["pred_home"].fillna(full_global_home))
    test_away_pred = clip_and_round(test_naive["pred_away"].fillna(full_global_away))

    final_artifacts["model"] = "Naive grouped mean"

print("Final model for submission:", final_artifacts["model"])

# 12. Test prediction & submission export

Model final menghasilkan prediksi di level match:
- `home_goals`
- `away_goals`

Lalu prediksi itu dikembalikan lagi ke format row-level:

- untuk baris home:
  - `team_goals = home_goals`
  - `opp_goals  = away_goals`
- untuk baris away:
  - `team_goals = away_goals`
  - `opp_goals  = home_goals`

Akhirnya file disimpan sebagai `submission.csv` dengan kolom:
- `Id`
- `team_goals`
- `opp_goals`

In [ ]:

# 12. Export submission.csv

test_submission = (
    test_match[["home_id", "away_id"]]
    .assign(home_goals_pred=test_home_pred, away_goals_pred=test_away_pred)
)

home_rows = test_submission.rename(columns={
    "home_id": "Id",
    "home_goals_pred": "team_goals",
    "away_goals_pred": "opp_goals",
})[["Id", "team_goals", "opp_goals"]]

away_rows = test_submission.rename(columns={
    "away_id": "Id",
    "away_goals_pred": "team_goals",
    "home_goals_pred": "opp_goals",
})[["Id", "team_goals", "opp_goals"]]

submission = pd.concat([home_rows, away_rows], axis=0, ignore_index=True)

# pastikan urutannya mengikuti test.csv asli
submission = test[["Id"]].merge(submission, on="Id", how="left")

assert submission.shape[0] == test.shape[0], "Jumlah row submission tidak sama dengan test."
assert submission["Id"].equals(test["Id"]), "Urutan Id submission tidak sesuai urutan test asli."
assert submission[["team_goals", "opp_goals"]].notna().all().all(), "Masih ada prediksi yang kosong."

submission["team_goals"] = submission["team_goals"].astype(int)
submission["opp_goals"] = submission["opp_goals"].astype(int)

submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("submission.csv berhasil disimpan ke:", submission_path.resolve())
display(submission.head(10))

# 13. Kesimpulan eksperimen

EXP 01 ini sengaja difokuskan sebagai baseline yang sehat dulu.

## Apa yang sudah dicapai
- Transformasi **2 row per match -> 1 row per match** sudah dibuat konsisten dan diberi sanity check.
- Modeling final **hanya memakai common features** antara train dan test.
- Validasi memakai **strict time-based holdout**, jadi lebih realistis dibanding random split.
- Notebook sudah menulis **`submission.csv`** dalam format yang siap dipakai.

## Batasan baseline ini
- Banyak fitur kuat di train memang belum dipakai, karena tidak tersedia di test.
- Distribusi test yang lebih modern kemungkinan masih menyisakan shift yang cukup besar.
- Kasus skor ekstrem belum ditangani secara khusus di baseline ini.

## Arah eksperimen berikutnya
Setelah baseline aman ini beres, eksperimen berikutnya bisa mulai mengeksplor:
- pembobotan era / recency,
- split atau penanganan khusus berdasarkan domain tertentu,
- objective / post-processing yang lebih cocok untuk target skor,
- dan struktur validasi yang lebih kaya.